In [16]:
# Double Q-Learning with Improved Prioritized Experience Replay for Maze

import numpy as np
import random
from collections import deque

# ─── Map & parameters ─────────────────────────────────────────
WIDTH, HEIGHT = 21, 11
NUM_POS = WIDTH * HEIGHT           # 231

treasure_list = [(0, 6), (3, 16), (8, 2), (10, 2), (10, 17)]
walls_coords = [
    (0,4),(0,5),(0,7),(0,9),(1,1),(1,2),(1,4),(1,9),(1,10),(1,14),(1,18),
    (2,1),(2,3),(2,5),(2,7),(2,8),(2,9),(2,11),(2,13),(2,15),(2,16),(2,17),(2,19),
    (3,2),(3,8),(3,11),(3,17),(4,1),(4,4),(4,6),(4,10),(4,13),(4,16),(4,17),(4,18),(4,20),
    (5,4),(5,5),(5,6),(5,8),(5,9),(5,14),(5,15),(6,1),(6,2),(6,3),(6,6),(6,8),(6,10),(6,15),(6,16),(6,17),(6,19),
    (7,4),(7,6),(7,8),(7,10),(7,11),(7,17),(7,19),(8,1),(8,4),(8,8),(8,10),(8,13),(8,15),(8,18),(8,19),
    (9,1),(9,2),(9,4),(9,6),(9,7),(9,17),(10,1),(10,4),(10,16),(10,19)
]
treasures = set(treasure_list)
walls = set(walls_coords)
START, GOAL = (0,0), (20,10)

# Q-learning parameters
MAX_EPISODES = 1000
MAX_STEPS    = 1000
EPSILON, EPS_DECAY, EPS_MIN = 0.9, 0.99, 0.01
ALPHA, GAMMA = 0.7, 0.995

# Prioritized Experience Replay
BUFFER_CAPACITY = 5000
REPLAY_BATCH    = 32
PER_EPSILON     = 1e-5
PER_UPDATE_FREQ = 50

transition_buffer = deque(maxlen=BUFFER_CAPACITY)
priority_buffer   = deque(maxlen=BUFFER_CAPACITY)

# Early stop
patience, no_improve = 500, 0

ACTIONS = ['up','down','left','right']
NUM_ACTIONS = len(ACTIONS)
NUM_MASK = 1 << len(treasure_list)  # 32
NUM_STATES = NUM_POS * NUM_MASK     # 7392

# Double Q tables
Q1 = np.zeros((NUM_STATES, NUM_ACTIONS))
Q2 = np.zeros((NUM_STATES, NUM_ACTIONS))

# Utility functions
def to_index(pos): return pos[1] * WIDTH + pos[0]
def to_pos(idx): return (idx % WIDTH, idx // WIDTH)
def encode_state(pos, mask): return to_index(pos) * NUM_MASK + mask

def compute_target(pos, act):
    x,y = pos
    if act=='up':    y = max(0,   y-1)
    elif act=='down':y = min(HEIGHT-1, y+1)
    elif act=='left':x = max(0,   x-1)
    else:            x = min(WIDTH-1,  x+1)
    return (x,y)

def nearest_target(pos, mask):
    rem = [t for i,t in enumerate(treasure_list) if not (mask & (1<<i))]
    return min(rem, key=lambda t: abs(t[0]-pos[0])+abs(t[1]-pos[1])) if rem else GOAL

def manhattan(a,b): return abs(a[0]-b[0]) + abs(a[1]-b[1])

# Training loop
best_steps, best_score, best_path = MAX_STEPS+1, -1, None
epsilon = EPSILON

for ep in range(1, MAX_EPISODES+1):
    pos, mask = START, 0
    state = encode_state(pos, mask)
    path = [state]; score = 0; improved=False

    for step in range(1, MAX_STEPS+1):
        # ε-greedy with obstacle avoidance
        while True:
            if random.random() < epsilon:
                a = random.randrange(NUM_ACTIONS)
            else:
                a = np.argmax(Q1[state] + Q2[state])
            new_pos = compute_target(pos, ACTIONS[a])
            if new_pos in walls or new_pos==pos: continue
            break

        # get reward
        reward = -1
        if new_pos in treasures:
            i = treasure_list.index(new_pos)
            if not mask & (1<<i):
                mask |= (1<<i)
                reward = +10; score +=1
            else: reward = -1
        elif new_pos==GOAL:
            reward = +58 if mask==(NUM_MASK-1) else -50

        # reward shaping
        tgt=nearest_target(pos, mask)
        d0=manhattan(pos,tgt); d1=manhattan(new_pos,tgt)
        reward += +2 if d1<d0 else -2 if d1>d0 else 0

        new_state = encode_state(new_pos, mask)

        # Double Q update
        if random.random()<0.5:
            a_next = np.argmax(Q1[new_state])
            target_q = reward + GAMMA * Q2[new_state, a_next]
            td_err = target_q - Q1[state,a]
            Q1[state,a] += ALPHA * td_err
        else:
            a_next = np.argmax(Q2[new_state])
            target_q = reward + GAMMA * Q1[new_state, a_next]
            td_err = target_q - Q2[state,a]
            Q2[state,a] += ALPHA * td_err

        # add to PER buffer
        transition_buffer.append((state,a,reward,new_state))
        priority_buffer.append(abs(td_err)+PER_EPSILON)

        # PER update every few steps
        if step % PER_UPDATE_FREQ == 0 and len(transition_buffer)>=REPLAY_BATCH:
            p = np.array(priority_buffer,float)
            p /= p.sum()
            idxs = np.random.choice(len(p),REPLAY_BATCH,p=p)
            for idx in idxs:
                s,a0,r0,sn = transition_buffer[idx]
                if random.random()<0.5:
                    an = np.argmax(Q1[sn])
                    t_q = r0 + GAMMA*Q2[sn,an]
                    Q1[s,a0] += ALPHA*(t_q-Q1[s,a0])
                else:
                    an = np.argmax(Q2[sn])
                    t_q = r0 + GAMMA*Q1[sn,an]
                    Q2[s,a0] += ALPHA*(t_q-Q2[s,a0])

        pos, state = new_pos, new_state
        path.append(state)

        if new_pos==GOAL and mask==(NUM_MASK-1):
            improved=True
            break

    # decay epsilon
    epsilon = max(EPS_MIN, epsilon*EPS_DECAY)

    # record best
    if improved and step<best_steps:
        best_steps, best_score, best_path = step, score, path.copy()

    # early stop
    no_improve = 0 if improved else no_improve+1
    if epsilon<=EPS_MIN and no_improve>=patience: break

    if ep%100==0:
        print(f"Ep{ep:4d} ε={epsilon:.3f} best_steps={best_steps}")

# Results output omitted for brevity

#%% [markdown]  
# ## 輸出 & 儲存結果  

#%%  
if best_path is None:  
    print("未找到收集 5 寶並到終點的路徑")  
else:  
    print("=== 最佳結果 ===")  
    print(f"步數：{best_steps}，寶藏：{best_score}（應為 {len(treasure_list)}）")  
    np.save('q1.npy', Q1)  
    np.save('q2.npy', Q2)  
    # 文字化路徑顯示  
    maze = [[' ']*WIDTH for _ in range(HEIGHT)]  
    for x,y in walls_coords:  maze[y][x]='X'  
    for x,y in treasure_list: maze[y][x]='O'  
    maze[START[1]][START[0]]='S'; maze[GOAL[1]][GOAL[0]]='G'  
    for s in best_path:  
        x,y = to_pos(s//NUM_MASK)  
        if maze[y][x]==' ': maze[y][x]='.'  
    print("\n路徑（. 為走過的路徑）：")  
    for row in maze:  
        print(''.join(row))

Ep 100 ε=0.329 best_steps=1001
Ep 200 ε=0.121 best_steps=1001
Ep 300 ε=0.044 best_steps=1001
Ep 400 ε=0.016 best_steps=1001
未找到收集 5 寶並到終點的路徑
